# Bài học 11 - Giao thức Tác nhân-tới-Tác nhân (A2A)


## Cài đặt


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Giao thức A2A là gì?

The **Giao thức Tác nhân-đến-Tác nhân (A2A)** là một tiêu chuẩn mở cho phép các tác nhân AI giao tiếp,
khám phá lẫn nhau, và hợp tác — ngay cả khi chúng được xây dựng trên các framework khác nhau hoặc được lưu trữ bởi các dịch vụ khác nhau.

Key concepts:

- **Khám phá** – Các tác nhân xuất bản một *Thẻ tác nhân* mô tả khả năng của họ, giúp
  dễ dàng cho các tác nhân khác (hoặc bộ điều phối) tìm đúng chuyên gia cho một nhiệm vụ.
- **Truyền thông điệp** – Các tác nhân trao đổi các thông điệp có cấu trúc qua một giao thức chung, nên
  một yêu cầu từ một tác nhân có thể được hiểu và được thực hiện bởi tác nhân khác bất kể
  triển khai nội bộ của nó.
- **Vòng đời nhiệm vụ** – A2A định nghĩa các trạng thái như *đã gửi*, *đang xử lý*, *hoàn thành*, và
  *thất bại*, cung cấp cho bộ điều phối tầm nhìn đầy đủ về cách một nhiệm vụ được ủy quyền đang tiến triển.

Trong bài học này chúng tôi mô phỏng hợp tác theo phong cách A2A bằng cách kết nối ba tác nhân du lịch chuyên môn vào một luồng công việc nơi mỗi tác nhân đóng góp chuyên môn của mình và chuyển kết quả cho tác nhân tiếp theo.


## Tạo các đại lý du lịch chuyên biệt


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Hợp tác đa tác nhân thông qua luồng công việc

Chúng tôi kết nối ba tác nhân vào một luồng công việc tuần tự phản chiếu việc truyền tin A2A:

1. **CurrencyExchangeAgent** nhận yêu cầu của người dùng và đưa ra hướng dẫn về tiền tệ.
2. **ActivityPlannerAgent** nhận bối cảnh đã được làm phong phú và thêm các đề xuất hoạt động.
3. **TravelManagerAgent** tổng hợp cả hai đầu vào thành một bản tóm tắt chuyến đi cuối cùng.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Hiểu A2A trong môi trường sản xuất

Trong một môi trường sản xuất, giao thức A2A mở khóa các kịch bản liên dịch vụ mạnh mẽ:

| Capability | Description |
|---|---|
| **Cross-framework interop** | Một agent được xây dựng với một framework có thể ủy quyền nhiệm vụ cho một agent được xây dựng bằng bất kỳ framework nào khác tương thích với A2A, cho phép khả năng tương tác thực sự giữa các tổ chức. |
| **Service boundaries** | Các agent có thể tồn tại trong các microservice riêng biệt, vùng đám mây khác nhau, hoặc thậm chí các tổ chức khác nhau trong khi vẫn hợp tác một cách liền mạch. |
| **Dynamic discovery** | Một bộ điều phối có thể truy vấn một registry Agent Card tại thời gian chạy để tìm chuyên gia phù hợp nhất cho một tác vụ phụ cụ thể. |
| **Streaming & push notifications** | A2A hỗ trợ Server-Sent Events (SSE) để cập nhật tiến độ theo thời gian thực và thông báo đẩy cho các tác vụ chạy lâu. |

The workflow we built above is a simplified, in-process version of this pattern. In a real
deployment each agent would expose an HTTP endpoint, publish an Agent Card, and communicate
via the A2A JSON-RPC protocol.


## Tóm tắt

Trong bài học này bạn đã học:

1. **Giao thức A2A là gì** — một tiêu chuẩn mở cho khám phá giữa các đại lý, nhắn tin,
   và quản lý nhiệm vụ.
2. **Cách tạo các đại lý chuyên biệt** — một đại lý Chuyển đổi Tiền tệ, một đại lý Lập kế hoạch Hoạt động,
   và một bộ điều phối Quản lý Du lịch.
3. **Cách kết nối các đại lý vào một luồng công việc** — sử dụng `WorkflowBuilder` để mô hình hóa việc truyền
   tin tuần tự giữa các đại lý.
4. **Cách A2A hoạt động trong môi trường sản xuất** — cho phép hợp tác giữa các framework và dịch vụ khác nhau
   với khả năng khám phá động và cập nhật liên tục theo luồng.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Miễn trừ trách nhiệm:
Văn bản này được dịch bằng dịch vụ dịch máy AI Co-op Translator (https://github.com/Azure/co-op-translator). Mặc dù chúng tôi nỗ lực đảm bảo độ chính xác, xin lưu ý rằng các bản dịch tự động có thể chứa lỗi hoặc không chính xác. Văn bản gốc bằng ngôn ngữ gốc nên được coi là nguồn có thẩm quyền. Đối với những thông tin quan trọng, khuyến nghị sử dụng dịch vụ dịch thuật chuyên nghiệp do con người thực hiện. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
